# NBA 3.1 Feature Engineering: TRx Features (Notebook 28)

**Purpose:** Compute HCP-level TRx (total prescription) features using V6 LAAD LRx table with all 11 pivot dimensions, v4 product group mapping, 28-cohort pipeline, and T1D/HF expansion.

**11 pivot dimensions (all from V6, all kept) — same as Notebook 26 but with TRx metric.**

**Data source:** V6 LAAD LRx table (same as NBA 3.0)
**Computation engine:** step2_fill_missing + step3_lag from 00_step23_function
**Output:** `kerendia_nba_3_1_volume/notebook_output/hcp_features/trx_features/`

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_step23_function"

In [0]:
# Cell 1: Configuration
import gc
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, expr, concat, substring, trunc, date_format

METRIC = "TRX"
V6_TABLE = "KERENDIA_LAAD_BASED_LRX_V6"
VOLUME_BASE = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
PRODUCT_MAPPING_PATH = f"{VOLUME_BASE}/final_product_mapping"
HCP_UNIVERSE_PATH = f"{VOLUME_BASE}/HCP_Universe_without_call_hcps"
COHORTS_PATH = f"{VOLUME_BASE}/cohorts/claims/cohorts.parquet"
V4_MAPPING_PATH = "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/Product_Group_Mapping/Kerendia_Market_Basket_Product_Group_Mapping_and_Priority_v4.csv"
OUTPUT_PATH = f"{VOLUME_BASE}/notebook_output/hcp_features/trx_features"
LAG_WINDOWS_FULL = [1, 3, 6, 12]
LAG_WINDOWS_REDUCED = [1, 12]
DROP_GROUPS = ["BEXAGLIFLOZIN", "ADLYXIN", "CYCLOSET", "MIGLITOL", "TEKTURNA", "ACTOS", "PIOGLITAZONE_ACTOS COMBOS", "SULFONYLUREAS"]
PIVOT_DIMENSIONS = ["PRODUCT_GROUP", "SPECIALTY_GROUP", "SWITCH_FROM", "ADD_ON_TO", "AGE_GROUP", "PAY_TYPE", "SOURCE_OF_BUSINESS", "RX_CHANNEL", "INDICATION_GROUP", "HYPERTENSION_FLAG", "GENDER"]
DIM_PREFIXES = {"PRODUCT_GROUP": "TOTAL_HCP_TRX_CLAIMS_", "SPECIALTY_GROUP": "TOTAL_HCP_TRX_CLAIMS_SPECIALTY_GROUP_", "SWITCH_FROM": "TOTAL_HCP_TRX_CLAIMS_SWITCH_FROM_", "ADD_ON_TO": "TOTAL_HCP_TRX_CLAIMS_ADD_ON_TO_", "AGE_GROUP": "TOTAL_HCP_TRX_CLAIMS_AGE_GROUP_", "PAY_TYPE": "TOTAL_HCP_TRX_CLAIMS_PAY_TYPE_", "SOURCE_OF_BUSINESS": "TOTAL_HCP_TRX_CLAIMS_SOB_", "RX_CHANNEL": "TOTAL_HCP_TRX_CLAIMS_CHANNEL_", "INDICATION_GROUP": "TOTAL_HCP_TRX_CLAIMS_INDICATION_GROUP_", "HYPERTENSION_FLAG": "TOTAL_HCP_TRX_CLAIMS_HYPERTENSION_FLAG_", "GENDER": "TOTAL_HCP_TRX_CLAIMS_GENDER_"}
from pyspark.storagelevel import StorageLevel

dbutils.widgets.text("test_mode", "false")
TEST_MODE = dbutils.widgets.get("test_mode") == "true"
TEST_HCP_COUNT = 1000
print(f"Metric: {METRIC}, Dimensions: {len(PIVOT_DIMENSIONS)}, Output: {OUTPUT_PATH}")
print(f"Test mode: {TEST_MODE}, Test HCP count: {TEST_HCP_COUNT}")

In [0]:
# Cell 2: Load product mappings
import pandas as pd
df_v4 = pd.read_csv(V4_MAPPING_PATH)
print(f"v4 mapping: {len(df_v4)} rows, {df_v4['PRODUCT_GROUP'].nunique()} groups")
priority_map = df_v4.groupby('PRODUCT_GROUP')['PRIORITY'].min().to_dict()
keep_groups = set(df_v4[df_v4['PROPOSED_ACTION'].isin(['KEEP', 'RECLASSIFY', 'ADD'])]['PRODUCT_GROUP'].unique())
all_keep = keep_groups - set(DROP_GROUPS)
print(f"Product groups to include: {len(all_keep)}")
df_dx_mapping = pd.read_csv(f"{PRODUCT_MAPPING_PATH}/Kerendia_Dx_Final_Product_Mapping.csv")
df_dx_flags = pd.read_csv(f"{PRODUCT_MAPPING_PATH}/Kerendia_Dx_Flag_Additions.csv")
t1d_codes = df_dx_mapping[df_dx_mapping['DM_TYPE'] == 'T1D']['MATCH_VALUE'].tolist()
print(f"T1D diagnosis codes: {len(t1d_codes)} codes")

In [0]:
# Cell 3: Read V6, filter to HCP universe, DROP exclusion, date processing
# OPTIMIZED: date filter in SQL, broadcast join, cache, minimal count() calls
hcp_universe = spark.read.parquet(HCP_UNIVERSE_PATH).select("PROVIDER_ID")
if TEST_MODE:
    hcp_universe = hcp_universe.limit(TEST_HCP_COUNT)
    print(f"TEST MODE: limiting to {TEST_HCP_COUNT} HCPs")

query = f"""
    SELECT DISTINCT WEEK_ENDING_DATE, PROVIDER_ID, PERIOD_ID, PERIOD_TYPE, SPECIALTY_GROUP,
    PRODUCT_GROUP, SWITCH_FROM, ADD_ON_TO, AGE_GROUP, GENDER, PAY_TYPE, SOURCE_OF_BUSINESS,
    RX_CHANNEL, INDICATION_GROUP, HYPERTENSION_FLAG, PAYER_PLAN_ID, NBRX, TRX, NRX, NPI, PRI_SPCL_DESC
    from `ph-com-ai-us-prod-catalog-2553213839599380`.`kerendia_laad_based_lrx`.{V6_TABLE}
"""
hcp_lrx_claims = spark.sql(query)
hcp_lrx_claims = F.broadcast(hcp_universe).join(hcp_lrx_claims, on="PROVIDER_ID", how="inner")
hcp_lrx_claims = hcp_lrx_claims.filter(~col("PRODUCT_GROUP").isin(DROP_GROUPS))
hcp_lrx_claims = hcp_lrx_claims.withColumn("INDICATION_GROUP", F.when(col("INDICATION_GROUP") == "ALL OTHERS", "ALL_OTHERS").otherwise(col("INDICATION_GROUP")))
hcp_lrx_claims = hcp_lrx_claims.withColumn("SVC_DT_M", F.trunc(col("PERIOD_ID"), 'month')).drop('WEEK_ENDING_DATE', 'PERIOD_TYPE', 'PAYER_PLAN_ID', 'NPI', 'PRI_SPCL_DESC')
hcp_lrx_claims.persist(StorageLevel.MEMORY_AND_DISK)
materialized_count = hcp_lrx_claims.count()
print(f"Cached V6 (filtered): {materialized_count:,} rows, {len(hcp_lrx_claims.columns)} cols")
index_cols = ["PROVIDER_ID", "SVC_DT_M"]

In [0]:
# Cell 4: Load cohorts and cross-join
# OPTIMIZED: removed count() calls, broadcast, cache
df_cohorts = spark.read.parquet(COHORTS_PATH)
print(f"Cohorts: {df_cohorts.count()}")
df_hcp_cohorts = hcp_universe.crossJoin(F.broadcast(df_cohorts.select("cohort_id", "prediction_month", "lookback_start_date", "lookback_end_date")))
df_hcp_cohorts = df_hcp_cohorts.withColumn("HCP_COHORT_ID", F.concat_ws("_", F.col("PROVIDER_ID"), F.col("cohort_id")))
df_hcp_cohorts.persist(StorageLevel.MEMORY_AND_DISK)
print(f"HCP x Cohort cross-join cached")

In [0]:
# Cell 5: INDICATION_GROUP T1D augmentation
all_t1d_codes = list(set(df_dx_mapping[df_dx_mapping['DM_TYPE'] == 'T1D']['MATCH_VALUE'].tolist() + df_dx_flags[df_dx_flags['DM_TYPE'] == 'T1D']['MATCH_VALUE'].tolist()))
print(f"T1D codes: {len(all_t1d_codes)}")

In [0]:
# Cell 6: Pivot by ALL 11 V6 dimensions and combine into df_spine
def clean_col_name(c):
    return c.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()

def pivot_dimension(df, dim, prefix, metric_col="TRX"):
    df_clean = df.withColumn(dim, F.when((col(dim).isNull()) | (col(dim) == 'NULL'), 'UNKNOWN').otherwise(col(dim)))
    df_p = df_clean.groupBy(index_cols).pivot(dim).agg(F.sum(metric_col).alias(f"SUM_{metric_col}"))
    return df_p.select([F.col(c).alias(prefix + clean_col_name(c)) if c not in index_cols else col(c) for c in df_p.columns])

pivot_dfs = {}
for dim in PIVOT_DIMENSIONS:
    print(f"Pivoting by {dim}...")
    pivot_dfs[dim] = pivot_dimension(hcp_lrx_claims, dim, DIM_PREFIXES[dim])
df_spine = pivot_dfs[PIVOT_DIMENSIONS[0]]
for dim in PIVOT_DIMENSIONS[1:]:
    df_spine = df_spine.join(pivot_dfs[dim], how='left', on=index_cols)
del pivot_dfs; gc.collect()
print(f"df_spine built: {len(df_spine.columns)} cols")

In [0]:
# Cell 7: step2_fill_missing, step3_lag, priority pruning, filter, save
# OPTIMIZED: precompute pruning columns, minimal count() calls
pg_prefix = DIM_PREFIXES["PRODUCT_GROUP"]
cols_to_prune = []
for pg in all_keep:
    pri = priority_map.get(pg, 6)
    if pri > 4:
        pg_clean = pg.replace(", ", " ").replace("+", "and").replace("/", " or ").replace("-", "_").replace(" ", "_").replace("__", "_").replace("(", "").replace(")", "").replace("&", "").upper()
        col_base = f"{pg_prefix}{pg_clean}"
        cols_to_prune.extend([f"{col_base}_IN_LAST_3_MONTH", f"{col_base}_IN_LAST_6_MONTH", f"{col_base}_RUNNING_TOTAL"])

df_spine_step2 = step2_fill_missing(df_spine, key_col="PROVIDER_ID", date_col="SVC_DT_M")
keep_cols = ["PROVIDER_ID", "SVC_DT_M"]
cols_to_drop = [c for c in df_spine_step2.columns if c not in keep_cols]
df_feats = step3_lag(df_spine_step2, key_col="PROVIDER_ID", date_col="SVC_DT_M", last_i_month_list=[1, 3, 6, 12])
del df_spine_step2; gc.collect()
df_feats = df_feats.drop(*cols_to_drop).filter(df_feats.PROVIDER_ID.isNotNull())
cols_to_prune_existing = [c for c in cols_to_prune if c in df_feats.columns]
if cols_to_prune_existing:
    df_feats = df_feats.drop(*cols_to_prune_existing)
df_feats = df_feats.filter(F.to_date('SVC_DT_M') >= '2023-01-01')
del df_spine; gc.collect()
print(f"After step3_lag + pruning: {len(df_feats.columns)} cols")
df_feats = df_feats.withColumn("PULL_DT", F.lit(str(dt.today().strftime('%Y_%m_%d'))))
print(f"Saving to: {OUTPUT_PATH}")
df_feats.write.mode("overwrite").format("delta").option("overwriteSchema", "true").save(OUTPUT_PATH)
print(f"Notebook 28 ({METRIC}) complete. Saved to NBA 3.1 volume.")